In [5]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

I0000 00:00:1792311298.847912     488 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792311298.850620     488 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792311299.304626     488 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792311300.204764     488 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [6]:
TICKER   = "AAPL"
PERIOD   = "10y"        
SEQ_LEN  = 60          
TEST_SPLIT = 0.2       
EPOCHS   = 50
BATCH    = 32

In [7]:
def download_data(ticker: str, period: str) -> pd.DataFrame:
    df = yf.download(ticker, period=period, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):   # handle newer yfinance multi-index
        df.columns = df.columns.get_level_values(0)
    return df


In [8]:
def add_features(df):
    df = df.copy()
    df["ret_1"]  = df["Close"].pct_change()
    df["ma_5"]   = df["Close"].rolling(5).mean()
    df["ma_20"]  = df["Close"].rolling(20).mean()
    df["volatility"] = df["ret_1"].rolling(10).std()

    delta = df["Close"].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df["rsi"] = 100 - 100 / (1 + gain / loss)

    df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

    df = df.dropna()
    return df

In [9]:
def build_sequences(data, targets, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i + seq_len])
        y.append(targets[i + seq_len])
    return np.array(X), np.array(y)

In [10]:
def main():
    df = download_data(TICKER, PERIOD)
    df = add_features(df)

    features = ["Open", "High", "Low", "Close", "Volume",
                "ma_5", "ma_20", "volatility", "rsi"]
    X_raw = df[features].values
    y_raw = df["target"].values

    split_idx = int(len(X_raw) * (1 - TEST_SPLIT))
    scaler = MinMaxScaler()
    scaler.fit(X_raw[:split_idx])
    X_scaled = scaler.transform(X_raw)

    X, y = build_sequences(X_scaled, y_raw, SEQ_LEN)
    split = split_idx - SEQ_LEN
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, len(features))),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid")    
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy",
                  metrics=["accuracy"])
    model.summary()


    es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    history = model.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=EPOCHS, batch_size=BATCH,
        callbacks=[es], verbose=1
    )

    y_prob = model.predict(X_test).flatten()
    y_pred = (y_prob > 0.5).astype(int)

    print("\n--- Classification Report ---")
    print(classification_report(y_test, y_pred, target_names=["DOWN", "UP"]))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


In [11]:
if __name__ == "__main__":
    main()

[*********************100%***********************]  1 of 1 completed
E0000 00:00:1792311302.872575     488 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1792311302.872916     985 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1792311302.902825     488 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
/home/amirul/stock-market-prediction-with-portfolio-management-/env/lib/python3.13/site-packages/keras/sr

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 64)         │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,905 (124.63 KB)

 Trainable params: 31,905 (124.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.5246 - loss: 0.6938 - val_accuracy: 0.5636 - val_loss: 0.6869
Epoch 2/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.5319 - loss: 0.6916 - val_accuracy: 0.5636 - val_loss: 0.6924
Epoch 3/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.5319 - loss: 0.6917 - val_accuracy: 0.5636 - val_loss: 0.6887
Epoch 4/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.5313 - loss: 0.6916 - val_accuracy: 0.5636 - val_loss: 0.6909
Epoch 5/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5313 - loss: 0.6923 - val_accuracy: 0.5636 - val_loss: 0.6912
Epoch 6/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.5313 - loss: 0.6910 - val_accuracy: 0.5636 - val_loss: 0.6900
Epoch 7/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5313 - loss: 0.6908 - val_accuracy: 0.5636 - val_loss: 0.6924
Epoch 8/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.5301 - loss: 0.6912 - val_accuracy: 0.5636 - v

/home/amirul/stock-market-prediction-with-portfolio-management-/env/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/amirul/stock-market-prediction-with-portfolio-management-/env/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/amirul/stock-market-prediction-with-portfolio-management-/env/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples.